# Driver Drowsiness Detection — MobileNetV2 Transfer Learning

This notebook trains a **MobileNetV2 transfer learning model** (~2.4M trainable parameters) using ImageNet pretrained weights for drowsiness detection.

## Workflow
1. Configuration
2. Dataset loading
3. Build MobileNetV2 model
4. Phase 1 — Feature extraction (frozen base)
5. Phase 2 — Fine-tuning (unfreeze top layers)
6. Evaluate on test set
7. Compare with CNN from scratch

In [ ]:
import sys, os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_ROOT = '/content/drive/MyDrive/Driver-Drowsiness-Detection-System'
except ImportError:
    REPO_ROOT = os.path.abspath('..')

sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)
print('Working directory:', os.getcwd())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from pathlib import Path

from models.mobilenet_model import (
    build_mobilenet_model,
    get_mobilenet_callbacks,
    unfreeze_for_fine_tuning,
)
from utils.data_loader import create_data_generators
from utils.metrics import (
    plot_training_history,
    plot_confusion_matrix,
    print_classification_report,
    plot_model_comparison,
)

print(f'TensorFlow version: {tf.__version__}')
print(f'GPUs available: {len(tf.config.list_physical_devices("GPU"))}')

## 1. Configuration

In [ ]:
DATA_DIR   = 'data/'
IMAGE_SIZE = (96, 96)  # MobileNetV2 minimum is 32x32; 96x96 balances speed/accuracy
BATCH_SIZE = 32
EPOCHS_P1  = 20        # Feature extraction phase
EPOCHS_P2  = 15        # Fine-tuning phase
FINE_TUNE_AT = 100     # Unfreeze MobileNetV2 from this layer index
SEED = 42

tf.random.set_seed(SEED)
np.random.seed(SEED)

## 2. Dataset

In [ ]:
train_gen, val_gen, test_gen = create_data_generators(
    DATA_DIR, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE
)
print(f'Train: {train_gen.n}  Val: {val_gen.n}  Test: {test_gen.n}')

## 3. Build MobileNetV2 Model

In [ ]:
model, base_model = build_mobilenet_model(input_shape=(*IMAGE_SIZE, 3))
model.summary()

trainable_params = sum(tf.size(w).numpy() for w in model.trainable_weights)
total_params = model.count_params()
print(f'\nTotal parameters   : {total_params:,}')
print(f'Trainable (phase 1): {trainable_params:,}')

## 4. Phase 1 — Feature Extraction

In [ ]:
Path('saved_models').mkdir(exist_ok=True)
callbacks_p1 = get_mobilenet_callbacks('saved_models/mobilenet_phase1.weights.h5')

history_p1 = model.fit(
    train_gen,
    epochs=EPOCHS_P1,
    validation_data=val_gen,
    callbacks=callbacks_p1,
    verbose=1,
)
plot_training_history(history_p1, model_name='MobileNetV2 — Phase 1')

## 5. Phase 2 — Fine-Tuning

In [ ]:
unfreeze_for_fine_tuning(base_model, model, fine_tune_at=FINE_TUNE_AT)

trainable_params_ft = sum(tf.size(w).numpy() for w in model.trainable_weights)
print(f'Trainable parameters (fine-tuning): {trainable_params_ft:,}')

callbacks_p2 = get_mobilenet_callbacks('saved_models/mobilenet_best.weights.h5')

history_p2 = model.fit(
    train_gen,
    epochs=EPOCHS_P2,
    validation_data=val_gen,
    callbacks=callbacks_p2,
    verbose=1,
)
plot_training_history(history_p2, model_name='MobileNetV2 — Fine-Tuning')

## 6. Evaluate on Test Set

In [ ]:
test_gen.reset()
y_pred_proba = model.predict(test_gen, verbose=1).ravel()
y_true = test_gen.classes

metrics_mn = print_classification_report(y_true, y_pred_proba, model_name='MobileNetV2')
plot_confusion_matrix(
    y_true, (y_pred_proba >= 0.5).astype(int),
    model_name='MobileNetV2'
)

## 7. Model Comparison

In [ ]:
# These are the published results from the paper / training runs
results = {
    'CNN from Scratch': {
        'accuracy': 0.9899,
        'recall_drowsy': 0.98,
        'auc': 0.997,
    },
    'MobileNetV2': {
        'accuracy': metrics_mn['accuracy'],
        'recall_drowsy': metrics_mn['recall_drowsy'],
        'auc': metrics_mn['auc'],
    },
}

plot_model_comparison(results)

In [ ]:
model.save('saved_models/mobilenet_final.keras')
print('Model saved to saved_models/mobilenet_final.keras')